## My result

### result on O'Neil

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import math
from dataset import new_CombDataset
from collator import MDMSD_collate_fn
from model_test import MDMSD
from model_config import config
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support
)
from scipy.stats import pearsonr, spearmanr

# ============================================================
# Config
# ============================================================
cfg = config['combine']
os.environ["CUDA_VISIBLE_DEVICES"] = '4,5,6,7'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------- Manual Parameters ----------------
moa_csv = "datasets/moa/moa_test.csv"
syn_csv = "datasets/syn/my_oneil.csv"
cell_expr_csv = "datasets/ccle_expr_norm.csv"
emb_file1 = "datasets/ids/kpgt_base.npz"
emb_file2 = "datasets/ids/data_mol_repr_2000drug_conf1.npz"
ids_csv = "datasets/ids_smiles/ids_smiles.csv"
checkpoint = "checkpoints/best_test_epoch111_descAcc0.92_synMAE8.11.pt"

selected_desc_labels = cfg.get("selected_desc_labels", None)
selected_metrics = cfg.get("selected_metrics", None)

# ============================================================
# Dataset & Loader
# ============================================================
test_dataset = new_CombDataset(
    moa_csv=moa_csv,
    syn_csv=syn_csv,
    cell_expr_csv=cell_expr_csv,
    emb_file1=emb_file1,
    emb_file2=emb_file2,
    ids_csv=ids_csv,
    selected_desc_labels=selected_desc_labels,
    selected_metrics=selected_metrics
)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg['batch_size'],
    shuffle=False,
    collate_fn=MDMSD_collate_fn,
    num_workers=cfg.get("num_workers", 4),
    pin_memory=True
)

# ============================================================
# Model
# ============================================================
model = MDMSD(
    k_embed=cfg['k_embed'],
    u_embed=cfg['u_embed'],
    num_desc_classes=cfg['num_desc_classes'],
    num_action_classes=cfg['num_action_classes'],
    cell_expr_dim=cfg['cell_expr_dim'],
    dimreduct_dim=cfg['dimreduct_dim'],
    expr_hidden=cfg['expr_hidden'],
    regress_hidden=cfg['regress_hidden'],
    lstm_hdims=cfg['lstm_hdims'],
    dropout_atten=cfg['dropout_atten'],
    dropout_dimreduct=cfg['dropout_dimreduct'],
    dropout_cellfcn=cfg['dropout_cellfcn'],
    dropout_downstream=cfg['dropout_downstream'],
    num_heads=cfg['num_heads'],
    mode=cfg['mode'],
    use_bn=cfg['use_bn']
)

if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
model.to(device)

# ============================================================
# Load Checkpoint (safe loading)
# ============================================================
print(f"Loading checkpoint: {checkpoint}")
state_dict = torch.load(checkpoint, map_location=device)
if any(k.startswith("module.") for k in state_dict.keys()):
    print("Detected multi-GPU checkpoint, removing 'module.' prefixes…")
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

if hasattr(model, "module"):
    model.module.load_state_dict(state_dict, strict=False)
else:
    model.load_state_dict(state_dict, strict=False)
print("✅ Model loaded successfully.\n")
model.eval()

# ============================================================
# Test Loop
# ============================================================
all_desc_preds, all_desc_labels = [], []
all_action_preds, all_action_labels = [], []
all_syn_preds, all_syn_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", ncols=None):
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        (desc_logits, action_logits), syn_pred = model(
            batch_device["k_moa1"], batch_device["u_moa1"],
            batch_device["k_moa2"], batch_device["u_moa2"],
            batch_device["k_syn1"], batch_device["u_syn1"],
            batch_device["k_syn2"], batch_device["u_syn2"],
            batch_device["cell_expr"]
        )

        all_desc_preds.append(torch.argmax(desc_logits, dim=-1).cpu().numpy())
        all_desc_labels.append(batch_device["desc_label"].cpu().numpy())
        all_action_preds.append(torch.argmax(action_logits, dim=-1).cpu().numpy())
        all_action_labels.append(batch_device["action_label"].cpu().numpy())
        all_syn_preds.append(syn_pred.squeeze(-1).cpu().numpy())
        all_syn_labels.append(batch_device["syn_score"][:, 0].cpu().numpy())

desc_labels_all = np.concatenate(all_desc_labels)
desc_preds_all = np.concatenate(all_desc_preds)
action_labels_all = np.concatenate(all_action_labels)
action_preds_all = np.concatenate(all_action_preds)
syn_labels_all = np.concatenate(all_syn_labels)
syn_preds_all = np.concatenate(all_syn_preds)

# ============================================================
# Classification Metrics
# ============================================================
def per_class_metrics(y_true, y_pred, task_name):
    """calculate Accuracy, Precision, Recall, F1 for each class"""
    unique_classes = np.unique(y_true)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=unique_classes, zero_division=0
    )
    results = []
    for cls, p, r, f, s in zip(unique_classes, precision, recall, f1, support):
        mask = y_true == cls
        acc = accuracy_score(y_true[mask], y_pred[mask])
        results.append({
            "task": task_name,
            "class": int(cls),
            "accuracy": acc,
            "precision": p,
            "recall": r,
            "f1": f,
            "support": s
        })
    return pd.DataFrame(results)

desc_df = per_class_metrics(desc_labels_all, desc_preds_all, "Description")
action_df = per_class_metrics(action_labels_all, action_preds_all, "Action")

desc_acc = accuracy_score(desc_labels_all, desc_preds_all)
action_acc = accuracy_score(action_labels_all, action_preds_all)

desc_precision, desc_recall, desc_f1, _ = precision_recall_fscore_support(
    desc_labels_all, desc_preds_all, average='macro', zero_division=0
)
action_precision, action_recall, action_f1, _ = precision_recall_fscore_support(
    action_labels_all, action_preds_all, average='macro', zero_division=0
)

# ============================================================
# Regression Metrics
# ============================================================
def compute_mae_mse_rmse(target, pred):
    err = np.asarray(target) - np.asarray(pred)
    mae = float(np.mean(np.abs(err)))
    mse = float(np.mean(err ** 2))
    rmse = float(np.sqrt(mse))
    return mae, mse, rmse

def compute_r2(x, y):
    try:
        r = np.corrcoef(x, y)[0, 1]
        return r ** 2
    except Exception:
        return 0.0

mae, mse, rmse = compute_mae_mse_rmse(syn_labels_all, syn_preds_all)
r2 = compute_r2(syn_labels_all, syn_preds_all)
pearson_corr, _ = pearsonr(syn_labels_all, syn_preds_all)
spearman_corr, _ = spearmanr(syn_labels_all, syn_preds_all)

# ============================================================
# Print Summary
# ============================================================
print("=" * 70)
print(f"Description  Acc={desc_acc:.4f}  Prec={desc_precision:.4f}  Rec={desc_recall:.4f}  F1={desc_f1:.4f}")
print(f"Action       Acc={action_acc:.4f}  Prec={action_precision:.4f}  Rec={action_recall:.4f}  F1={action_f1:.4f}")
print("--------------------------------------------------------------")
print(f"Synergy MAE={mae:.4f}, MSE={mse:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, "
      f"Pearson={pearson_corr:.4f}, Spearman={spearman_corr:.4f}")
print("=" * 70)

print("\nPer-class Description Metrics:")
print(desc_df.to_string(index=False, justify="center"))
print("\nPer-class Action Metrics:")
print(action_df.to_string(index=False, justify="center"))


Loading checkpoint: checkpoints/best_test_epoch111_descAcc0.92_synMAE8.11.pt
✅ Model loaded successfully.



Testing:   0%|          | 0/27 [00:00<?, ?it/s]/root/miniconda3/envs/KPGT/lib/python3.7/site-packages/torch/nn/modules/rnn.py:692: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at  /opt/conda/conda-bld/pytorch_1634272168290/work/aten/src/ATen/native/cudnn/RNN.cpp:925.)
  self.dropout, self.training, self.bidirectional, self.batch_first)
Testing: 100%|██████████| 27/27 [00:16<00:00,  1.65it/s]


Description  Acc=0.9226  Prec=0.8813  Rec=0.8991  F1=0.8896
Action       Acc=0.9239  Prec=0.9189  Rec=0.9185  F1=0.9187
--------------------------------------------------------------
Synergy MAE=8.1089, MSE=123.7654, RMSE=11.1250, R²=0.1635, Pearson=0.4043, Spearman=0.3932

Per-class Description Metrics:
    task     class  accuracy  precision  recall     f1     support
Description   0     0.932011  0.951262  0.932011 0.941538   7560  
Description   1     0.955047  0.925987  0.955047 0.940292   2358  
Description   2     0.921041  0.935009  0.921041 0.927972   2343  
Description   3     0.788301  0.712846  0.788301 0.748677   1077  

Per-class Action Metrics:
 task   class  accuracy  precision  recall     f1     support
Action   0     0.897050  0.899035  0.897050 0.898041   4983  
Action   1     0.939916  0.938680  0.939916 0.939298   8355  


### Results on syn_test

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import math
from dataset import new_CombDataset
from collator import MDMSD_collate_fn
from model_test import MDMSD
from model_config import config
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support
)
from scipy.stats import pearsonr, spearmanr

# ============================================================
# Config
# ============================================================
cfg = config['combine']
os.environ["CUDA_VISIBLE_DEVICES"] = '2,3'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------- Manual Parameters ----------------
moa_csv = "datasets/moa/moa_test.csv"
syn_csv = "datasets/syn/syn_test.csv"
cell_expr_csv = "datasets/ccle_expr_norm.csv"
emb_file1 = "datasets/ids/kpgt_base.npz"
emb_file2 = "datasets/ids/data_mol_repr_2000drug_conf1.npz"
ids_csv = "datasets/ids_smiles/ids_smiles.csv"
checkpoint = "checkpoints/new_split/original/best_descAcc0.98_actionAcc0.99_synMAE7.02.pt"

selected_desc_labels = cfg.get("selected_desc_labels", None)
selected_metrics = cfg.get("selected_metrics", None)

# ============================================================
# Dataset & Loader
# ============================================================
test_dataset = new_CombDataset(
    moa_csv=moa_csv,
    syn_csv=syn_csv,
    cell_expr_csv=cell_expr_csv,
    emb_file1=emb_file1,
    emb_file2=emb_file2,
    ids_csv=ids_csv,
    selected_desc_labels=selected_desc_labels,
    selected_metrics=selected_metrics
)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg['batch_size'],
    shuffle=False,
    collate_fn=MDMSD_collate_fn,
    num_workers=cfg.get("num_workers", 4),
    pin_memory=True
)

# ============================================================
# Model
# ============================================================
model = MDMSD(
    k_embed=cfg['k_embed'],
    u_embed=cfg['u_embed'],
    num_desc_classes=cfg['num_desc_classes'],
    num_action_classes=cfg['num_action_classes'],
    cell_expr_dim=cfg['cell_expr_dim'],
    dimreduct_dim=cfg['dimreduct_dim'],
    expr_hidden=cfg['expr_hidden'],
    regress_hidden=cfg['regress_hidden'],
    lstm_hdims=cfg['lstm_hdims'],
    dropout_atten=cfg['dropout_atten'],
    dropout_dimreduct=cfg['dropout_dimreduct'],
    dropout_cellfcn=cfg['dropout_cellfcn'],
    dropout_downstream=cfg['dropout_downstream'],
    num_heads=cfg['num_heads'],
    mode=cfg['mode'],
    use_bn=cfg['use_bn']
)

if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
model.to(device)

# ============================================================
# Load Checkpoint (safe loading)
# ============================================================
print(f"Loading checkpoint: {checkpoint}")
state_dict = torch.load(checkpoint, map_location=device)
if any(k.startswith("module.") for k in state_dict.keys()):
    print("Detected multi-GPU checkpoint, removing 'module.' prefixes…")
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

if hasattr(model, "module"):
    model.module.load_state_dict(state_dict, strict=False)
else:
    model.load_state_dict(state_dict, strict=False)
print("✅ Model loaded successfully.\n")
model.eval()

# ============================================================
# Test Loop
# ============================================================
all_desc_preds, all_desc_labels = [], []
all_action_preds, all_action_labels = [], []
all_syn_preds, all_syn_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", ncols=None):
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        (desc_logits, action_logits), syn_pred = model(
            batch_device["k_moa1"], batch_device["u_moa1"],
            batch_device["k_moa2"], batch_device["u_moa2"],
            batch_device["k_syn1"], batch_device["u_syn1"],
            batch_device["k_syn2"], batch_device["u_syn2"],
            batch_device["cell_expr"]
        )

        all_desc_preds.append(torch.argmax(desc_logits, dim=-1).cpu().numpy())
        all_desc_labels.append(batch_device["desc_label"].cpu().numpy())
        all_action_preds.append(torch.argmax(action_logits, dim=-1).cpu().numpy())
        all_action_labels.append(batch_device["action_label"].cpu().numpy())
        all_syn_preds.append(syn_pred.squeeze(-1).cpu().numpy())
        all_syn_labels.append(batch_device["syn_score"][:, 0].cpu().numpy())

desc_labels_all = np.concatenate(all_desc_labels)
desc_preds_all = np.concatenate(all_desc_preds)
action_labels_all = np.concatenate(all_action_labels)
action_preds_all = np.concatenate(all_action_preds)
syn_labels_all = np.concatenate(all_syn_labels)
syn_preds_all = np.concatenate(all_syn_preds)

# ============================================================
# Classification Metrics
# ============================================================
def per_class_metrics(y_true, y_pred, task_name):
    """calculate Accuracy, Precision, Recall, F1 for each class"""
    unique_classes = np.unique(y_true)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=unique_classes, zero_division=0
    )
    results = []
    for cls, p, r, f, s in zip(unique_classes, precision, recall, f1, support):
        mask = y_true == cls
        acc = accuracy_score(y_true[mask], y_pred[mask])
        results.append({
            "task": task_name,
            "class": int(cls),
            "accuracy": acc,
            "precision": p,
            "recall": r,
            "f1": f,
            "support": s
        })
    return pd.DataFrame(results)

desc_df = per_class_metrics(desc_labels_all, desc_preds_all, "Description")
action_df = per_class_metrics(action_labels_all, action_preds_all, "Action")

desc_acc = accuracy_score(desc_labels_all, desc_preds_all)
action_acc = accuracy_score(action_labels_all, action_preds_all)

desc_precision, desc_recall, desc_f1, _ = precision_recall_fscore_support(
    desc_labels_all, desc_preds_all, average='macro', zero_division=0
)
action_precision, action_recall, action_f1, _ = precision_recall_fscore_support(
    action_labels_all, action_preds_all, average='macro', zero_division=0
)

# ============================================================
# Regression Metrics
# ============================================================
def compute_mae_mse_rmse(target, pred):
    err = np.asarray(target) - np.asarray(pred)
    mae = float(np.mean(np.abs(err)))
    mse = float(np.mean(err ** 2))
    rmse = float(np.sqrt(mse))
    return mae, mse, rmse

def compute_r2(x, y):
    try:
        r = np.corrcoef(x, y)[0, 1]
        return r ** 2
    except Exception:
        return 0.0

mae, mse, rmse = compute_mae_mse_rmse(syn_labels_all, syn_preds_all)
r2 = compute_r2(syn_labels_all, syn_preds_all)
pearson_corr, _ = pearsonr(syn_labels_all, syn_preds_all)
spearman_corr, _ = spearmanr(syn_labels_all, syn_preds_all)

# ============================================================
# Print Summary
# ============================================================
print("=" * 70)
print(f"Description  Acc={desc_acc:.4f}  Prec={desc_precision:.4f}  Rec={desc_recall:.4f}  F1={desc_f1:.4f}")
print(f"Action       Acc={action_acc:.4f}  Prec={action_precision:.4f}  Rec={action_recall:.4f}  F1={action_f1:.4f}")
print("--------------------------------------------------------------")
print(f"Synergy MAE={mae:.4f}, MSE={mse:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, "
      f"Pearson={pearson_corr:.4f}, Spearman={spearman_corr:.4f}")
print("=" * 70)

print("\nPer-class Description Metrics:")
print(desc_df.to_string(index=False, justify="center"))
print("\nPer-class Action Metrics:")
print(action_df.to_string(index=False, justify="center"))


/root/miniconda3/envs/KPGT/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading checkpoint: checkpoints/new_split/original/best_descAcc0.98_actionAcc0.99_synMAE7.02.pt
✅ Model loaded successfully.



Testing:   0%|          | 0/60 [00:00<?, ?it/s]/root/miniconda3/envs/KPGT/lib/python3.7/site-packages/torch/nn/modules/rnn.py:692: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at  /opt/conda/conda-bld/pytorch_1634272168290/work/aten/src/ATen/native/cudnn/RNN.cpp:925.)
  self.dropout, self.training, self.bidirectional, self.batch_first)
Testing: 100%|██████████| 60/60 [00:09<00:00,  6.51it/s]


Description  Acc=0.9201  Prec=0.8816  Rec=0.8901  F1=0.8856
Action       Acc=0.9267  Prec=0.9236  Rec=0.9192  F1=0.9213
--------------------------------------------------------------
Synergy MAE=7.0575, MSE=119.8657, RMSE=10.9483, R²=0.5432, Pearson=0.7370, Spearman=0.6517

Per-class Description Metrics:
    task     class  accuracy  precision  recall     f1     support
Description   0     0.938020  0.944018  0.938020 0.941009  17312  
Description   1     0.923837  0.943178  0.923837 0.933407   5462  
Description   2     0.928399  0.922395  0.928399 0.925387   5377  
Description   3     0.770308  0.716679  0.770308 0.742527   2499  

Per-class Action Metrics:
 task   class  accuracy  precision  recall     f1     support
Action   0     0.889712  0.912138  0.889712 0.900786  11470  
Action   1     0.948749  0.935002  0.948749 0.941825  19180  


## Moa only

### On ONeil

In [ ]:
import os
import numpy as np
import pandas as pd
import argparse
import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm
import math
from model_config import config
from dataset import MoADataset
from collator import MoADDI_collator
from model_ablation import MoADDI
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support
)

# ============================================================
# Config
# ============================================================
cfg = config['combine']
os.environ["CUDA_VISIBLE_DEVICES"] = '2,3'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------- Manual Parameters ----------------
syn_csv = 'datasets/syn/my_oneil.csv'
moa_csv = "datasets/moa/moa_test.csv"
emb_file1 = "datasets/ids/kpgt_base.npz"
emb_file2 = "datasets/ids/data_mol_repr_2000drug_conf1.npz"
ids_csv = "datasets/ids_smiles/ids_smiles.csv"
checkpoint = "checkpoints/moaddi/best_descAcc1.00_actionAcc1.00_moaddi.pt"

selected_desc_labels = cfg.get("selected_desc_labels", None)

# ============================================================
# Dataset & Loader
# ============================================================
test_dataset = MoADataset(
    syn_csv=syn_csv,
    moa_csv=moa_csv,
    emb_file1=emb_file1,
    emb_file2=emb_file2,
    ids_csv=ids_csv,
    selected_desc_labels=selected_desc_labels
)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg['batch_size'],
    shuffle=False,
    collate_fn=MoADDI_collator,
    num_workers=cfg.get("num_workers", 4),
    pin_memory=True
)

# ============================================================
# Model
# ============================================================
model = MoADDI(
    k_embed=cfg['k_embed'],
    u_embed=cfg['u_embed'],
    num_desc_classes=cfg['num_desc_classes'],
    num_action_classes=cfg['num_action_classes'],
    dimreduct_dim=cfg['dimreduct_dim'],
    lstm_hdims=cfg['lstm_hdims'],
    dropout_atten=cfg['dropout_atten'],
    dropout_dimreduct=cfg['dropout_dimreduct'],
    num_heads=cfg['num_heads'],
    mode=cfg['mode'],
    use_bn=cfg['use_bn']
)

if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
model.to(device)

# ============================================================
# Load Checkpoint (safe loading)
# ============================================================
print(f"Loading checkpoint: {checkpoint}")
state_dict = torch.load(checkpoint, map_location=device)
if any(k.startswith("module.") for k in state_dict.keys()):
    print("Detected multi-GPU checkpoint, removing 'module.' prefixes…")
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

if hasattr(model, "module"):
    model.module.load_state_dict(state_dict, strict=False)
else:
    model.load_state_dict(state_dict, strict=False)
print("✅ Model loaded successfully.\n")
model.eval()

# ============================================================
# Test Loop
# ============================================================
all_desc_preds, all_desc_labels = [], []
all_action_preds, all_action_labels = [], []
all_syn_preds, all_syn_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", ncols=None):
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        (desc_logits, action_logits)= model(
            batch_device["k_moa1"], batch_device["u_moa1"],
            batch_device["k_moa2"], batch_device["u_moa2"]
        )

        all_desc_preds.append(torch.argmax(desc_logits, dim=-1).cpu().numpy())
        all_desc_labels.append(batch_device["desc_label"].cpu().numpy())
        all_action_preds.append(torch.argmax(action_logits, dim=-1).cpu().numpy())
        all_action_labels.append(batch_device["action_label"].cpu().numpy())

desc_labels_all = np.concatenate(all_desc_labels)
desc_preds_all = np.concatenate(all_desc_preds)
action_labels_all = np.concatenate(all_action_labels)
action_preds_all = np.concatenate(all_action_preds)


# ============================================================
# Classification Metrics
# ============================================================
def per_class_metrics(y_true, y_pred, task_name):
    """calculate Accuracy, Precision, Recall, F1 for each class"""
    unique_classes = np.unique(y_true)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=unique_classes, zero_division=0
    )
    results = []
    for cls, p, r, f, s in zip(unique_classes, precision, recall, f1, support):
        mask = y_true == cls
        acc = accuracy_score(y_true[mask], y_pred[mask])
        results.append({
            "task": task_name,
            "class": int(cls),
            "accuracy": acc,
            "precision": p,
            "recall": r,
            "f1": f,
            "support": s
        })
    return pd.DataFrame(results)

desc_df = per_class_metrics(desc_labels_all, desc_preds_all, "Description")
action_df = per_class_metrics(action_labels_all, action_preds_all, "Action")

desc_acc = accuracy_score(desc_labels_all, desc_preds_all)
action_acc = accuracy_score(action_labels_all, action_preds_all)

desc_precision, desc_recall, desc_f1, _ = precision_recall_fscore_support(
    desc_labels_all, desc_preds_all, average='macro', zero_division=0
)
action_precision, action_recall, action_f1, _ = precision_recall_fscore_support(
    action_labels_all, action_preds_all, average='macro', zero_division=0
)


# ============================================================
# Print Summary
# ============================================================
print("=" * 70)
print(f"Description  Acc={desc_acc:.4f}  Prec={desc_precision:.4f}  Rec={desc_recall:.4f}  F1={desc_f1:.4f}")
print(f"Action       Acc={action_acc:.4f}  Prec={action_precision:.4f}  Rec={action_recall:.4f}  F1={action_f1:.4f}")
print("=" * 70)

print("\nPer-class Description Metrics:")
print(desc_df.to_string(index=False, justify="center"))
print("\nPer-class Action Metrics:")
print(action_df.to_string(index=False, justify="center"))

/root/miniconda3/envs/KPGT/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading checkpoint: checkpoints/moaddi/best_descAcc1.00_actionAcc1.00_moaddi.pt
✅ Model loaded successfully.



Testing:   0%|          | 0/27 [00:00<?, ?it/s]/root/miniconda3/envs/KPGT/lib/python3.7/site-packages/torch/nn/modules/rnn.py:692: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at  /opt/conda/conda-bld/pytorch_1634272168290/work/aten/src/ATen/native/cudnn/RNN.cpp:925.)
  self.dropout, self.training, self.bidirectional, self.batch_first)
Testing: 100%|██████████| 27/27 [00:14<00:00,  1.81it/s]


Description  Acc=0.9235  Prec=0.8829  Rec=0.8984  F1=0.8902
Action       Acc=0.9329  Prec=0.9323  Rec=0.9236  F1=0.9276

Per-class Description Metrics:
    task     class  accuracy  precision  recall     f1     support
Description   0     0.934259  0.950990  0.934259 0.942550   7560  
Description   1     0.953774  0.930108  0.953774 0.941792   2358  
Description   2     0.923602  0.932759  0.923602 0.928158   2343  
Description   3     0.781801  0.717818  0.781801 0.748444   1077  

Per-class Action Metrics:
 task   class  accuracy  precision  recall     f1     support
Action   0     0.887016  0.930135  0.887016 0.908064   4983  
Action   1     0.960263  0.934428  0.960263 0.947170   8355  


In [ ]:
import os
import numpy as np
import pandas as pd
import argparse
import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm
import math
from model_config import config
from dataset import MoADataset
from collator import MoADDI_collator
from model_ablation import MoADDI
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support
)

# ============================================================
# Config
# ============================================================
cfg = config['combine']
os.environ["CUDA_VISIBLE_DEVICES"] = '4,5,6,7'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------- Manual Parameters ----------------
syn_csv = 'datasets/syn/my_oneil.csv'
moa_csv = "datasets/moa/moa_test.csv"
emb_file1 = "datasets/ids/kpgt_base.npz"
emb_file2 = "datasets/ids/data_mol_repr_2000drug_conf1.npz"
ids_csv = "datasets/ids_smiles/ids_smiles.csv"
checkpoint = "checkpoints/moaddi/best_descAcc0.99_actionAcc1.00_moaddi.pt"

selected_desc_labels = cfg.get("selected_desc_labels", None)

# ============================================================
# Dataset & Loader
# ============================================================
test_dataset = MoADataset(
    syn_csv=syn_csv,
    moa_csv=moa_csv,
    emb_file1=emb_file1,
    emb_file2=emb_file2,
    ids_csv=ids_csv,
    selected_desc_labels=selected_desc_labels
)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg['batch_size'],
    shuffle=False,
    collate_fn=MoADDI_collator,
    num_workers=cfg.get("num_workers", 4),
    pin_memory=True
)

# ============================================================
# Model
# ============================================================
model = MoADDI(
    k_embed=cfg['k_embed'],
    u_embed=cfg['u_embed'],
    num_desc_classes=cfg['num_desc_classes'],
    num_action_classes=cfg['num_action_classes'],
    dimreduct_dim=cfg['dimreduct_dim'],
    lstm_hdims=cfg['lstm_hdims'],
    dropout_atten=cfg['dropout_atten'],
    dropout_dimreduct=cfg['dropout_dimreduct'],
    num_heads=cfg['num_heads'],
    mode=cfg['mode'],
    use_bn=cfg['use_bn']
)

if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
model.to(device)

# ============================================================
# Load Checkpoint (safe loading)
# ============================================================
print(f"Loading checkpoint: {checkpoint}")
state_dict = torch.load(checkpoint, map_location=device)
if any(k.startswith("module.") for k in state_dict.keys()):
    print("Detected multi-GPU checkpoint, removing 'module.' prefixes…")
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

if hasattr(model, "module"):
    model.module.load_state_dict(state_dict, strict=False)
else:
    model.load_state_dict(state_dict, strict=False)
print("✅ Model loaded successfully.\n")
model.eval()

# ============================================================
# Test Loop
# ============================================================
all_desc_preds, all_desc_labels = [], []
all_action_preds, all_action_labels = [], []
all_syn_preds, all_syn_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", ncols=None):
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        (desc_logits, action_logits)= model(
            batch_device["k_moa1"], batch_device["u_moa1"],
            batch_device["k_moa2"], batch_device["u_moa2"]
        )

        all_desc_preds.append(torch.argmax(desc_logits, dim=-1).cpu().numpy())
        all_desc_labels.append(batch_device["desc_label"].cpu().numpy())
        all_action_preds.append(torch.argmax(action_logits, dim=-1).cpu().numpy())
        all_action_labels.append(batch_device["action_label"].cpu().numpy())

desc_labels_all = np.concatenate(all_desc_labels)
desc_preds_all = np.concatenate(all_desc_preds)
action_labels_all = np.concatenate(all_action_labels)
action_preds_all = np.concatenate(all_action_preds)


# ============================================================
# Classification Metrics
# ============================================================
def per_class_metrics(y_true, y_pred, task_name):
    """calculate Accuracy, Precision, Recall, F1 for each class"""
    unique_classes = np.unique(y_true)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=unique_classes, zero_division=0
    )
    results = []
    for cls, p, r, f, s in zip(unique_classes, precision, recall, f1, support):
        mask = y_true == cls
        acc = accuracy_score(y_true[mask], y_pred[mask])
        results.append({
            "task": task_name,
            "class": int(cls),
            "accuracy": acc,
            "precision": p,
            "recall": r,
            "f1": f,
            "support": s
        })
    return pd.DataFrame(results)

desc_df = per_class_metrics(desc_labels_all, desc_preds_all, "Description")
action_df = per_class_metrics(action_labels_all, action_preds_all, "Action")

desc_acc = accuracy_score(desc_labels_all, desc_preds_all)
action_acc = accuracy_score(action_labels_all, action_preds_all)

desc_precision, desc_recall, desc_f1, _ = precision_recall_fscore_support(
    desc_labels_all, desc_preds_all, average='macro', zero_division=0
)
action_precision, action_recall, action_f1, _ = precision_recall_fscore_support(
    action_labels_all, action_preds_all, average='macro', zero_division=0
)


# ============================================================
# Print Summary
# ============================================================
print("=" * 70)
print(f"Description  Acc={desc_acc:.4f}  Prec={desc_precision:.4f}  Rec={desc_recall:.4f}  F1={desc_f1:.4f}")
print(f"Action       Acc={action_acc:.4f}  Prec={action_precision:.4f}  Rec={action_recall:.4f}  F1={action_f1:.4f}")
print("=" * 70)

print("\nPer-class Description Metrics:")
print(desc_df.to_string(index=False, justify="center"))
print("\nPer-class Action Metrics:")
print(action_df.to_string(index=False, justify="center"))

/root/miniconda3/envs/KPGT/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading checkpoint: checkpoints/moaddi/best_descAcc0.99_actionAcc1.00_moaddi.pt
✅ Model loaded successfully.



Testing:   0%|          | 0/27 [00:00<?, ?it/s]/root/miniconda3/envs/KPGT/lib/python3.7/site-packages/torch/nn/modules/rnn.py:692: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at  /opt/conda/conda-bld/pytorch_1634272168290/work/aten/src/ATen/native/cudnn/RNN.cpp:925.)
  self.dropout, self.training, self.bidirectional, self.batch_first)
Testing: 100%|██████████| 27/27 [00:13<00:00,  2.06it/s]


Description  Acc=0.9274  Prec=0.8888  Rec=0.9003  F1=0.8943
Action       Acc=0.9306  Prec=0.9264  Rec=0.9251  F1=0.9257

Per-class Description Metrics:
    task     class  accuracy  precision  recall     f1     support
Description   0     0.937698  0.952695  0.937698 0.945137   7560  
Description   1     0.964377  0.927028  0.964377 0.945334   2358  
Description   2     0.929577  0.940415  0.929577 0.934965   2343  
Description   3     0.769731  0.734929  0.769731 0.751927   1077  

Per-class Action Metrics:
 task   class  accuracy  precision  recall     f1     support
Action   0     0.903472  0.910046  0.903472 0.906747   4983  
Action   1     0.946738  0.942677  0.946738 0.944703   8355  


## Synergy only

### On Oneil

In [ ]:
# test_manual.py
import os
import math
import numpy as np
import pandas as pd
import argparse
import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm

from model_config import config
from dataset import SynergyDataset
from collator import SynergyDDI_collator
from model_ablation import SynergyDDI

# ---------------- Config ----------------
cfg = config['combine']

# ---------------- Manual Parameters ----------------
syn_csv="datasets/syn/my_oneil.csv"
cell_expr_csv="datasets/ccle_expr_norm.csv"
emb_file1="datasets/ids/kpgt_base.npz"
emb_file2="datasets/ids/data_mol_repr_2000drug_conf1.npz"
ids_csv="datasets/ids_smiles/ids_smiles.csv"
selected_metrics=cfg.get("selected_metrics", None)
checkpoint = "/root/wd/Frame/draft_model/checkpoints/synddi/best_synMAE6.65_syn.pt"

# ---------------- Device ----------------
os.environ["CUDA_VISIBLE_DEVICES"] = '2,3'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



# ---------------- Dataset & Loader ----------------
test_dataset = SynergyDataset(
    syn_csv=syn_csv,
    cell_expr_csv=cell_expr_csv,
    emb_file1=emb_file1,
    emb_file2=emb_file2,
    ids_csv=ids_csv,
    selected_metrics=cfg.get("selected_metrics", None)
)

test_loader = DataLoader(
    test_dataset, batch_size=cfg['batch_size'],
    shuffle=False, collate_fn=SynergyDDI_collator,
    num_workers=cfg.get("num_workers", 4), pin_memory=True
)

# ---------------- Model ----------------
model = SynergyDDI(
    k_embed=cfg['k_embed'],
    u_embed=cfg['u_embed'],
    cell_expr_dim=cfg['cell_expr_dim'],
    dimreduct_dim=cfg['dimreduct_dim'],
    expr_hidden=cfg['expr_hidden'],
    regress_hidden=cfg['regress_hidden'],
    dropout_dimreduct=cfg['dropout_dimreduct'],
    dropout_cellfcn=cfg['dropout_cellfcn'],
    dropout_downstream=cfg['dropout_downstream'],
    mode=cfg['mode'],
    use_bn=cfg['use_bn']
)
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
model.to(device)

# Load checkpoint
state_dict = torch.load(checkpoint, map_location=device)
if hasattr(model, 'module'):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)
model.eval()

# ---------------- Metrics ----------------
all_syn_preds, all_syn_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", ncols=None):
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        syn_pred = model(
            batch_device["k_syn1"], batch_device["u_syn1"],
            batch_device["k_syn2"], batch_device["u_syn2"],
            batch_device["cell_expr"]
        )
        all_syn_preds.append(syn_pred.squeeze(-1).cpu().numpy())
        all_syn_labels.append(batch_device["syn_score"][:, 0].cpu().numpy())

# ---------------- Concatenate ----------------
syn_labels_all = np.concatenate(all_syn_labels)
syn_preds_all = np.concatenate(all_syn_preds)

# ---------------- Compute Metrics ----------------

def compute_mae_mse_rmse(target,prediction):
    error = []
    for i in range(len(target)):
        error.append(target[i] - prediction[i])
    squaredError = []
    absError = []
    for val in error:
        squaredError.append(val * val)  # target-prediction squared
        absError.append(abs(val))  # error absolute value
    mae=sum(absError)/len(absError)  # MAE
    mse=sum(squaredError)/len(squaredError)  # MSE
    RMSE=math.sqrt(mse)  # RMSE
    return mae,mse,RMSE

def compute_rsquared(X, Y):
    xBar = np.mean(X)
    yBar = np.mean(Y)
    SSR = 0
    varX = 0
    varY = 0
    for i in range(0, len(X)):
        diffXXBar = X[i] - xBar
        diffYYBar = Y[i] - yBar
        SSR += (diffXXBar * diffYYBar)
        varX += diffXXBar ** 2
        varY += diffYYBar ** 2

    SST = math.sqrt(varX * varY)
    r2=round((SSR / SST) ** 2,3)
    return r2


mae, mse, rmse = compute_mae_mse_rmse(syn_labels_all, syn_preds_all)
r2 = compute_rsquared(syn_labels_all, syn_preds_all)

#syn_mse = mean_squared_error(syn_labels_all, syn_preds_all)
#syn_mae = mean_absolute_error(syn_labels_all, syn_preds_all)
#syn_rmse = np.sqrt(syn_mse)
#syn_r2 = r2_score(syn_labels_all, syn_preds_all)
syn_ccp, _ = pearsonr(syn_labels_all, syn_preds_all)
syn_ccs, _ = spearmanr(syn_labels_all, syn_preds_all)

print(f"Syn MAE={mae:.4f}, MSE = {mse:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}, CCp={syn_ccp:.4f}, CCs={syn_ccs:.4f}")


Testing: 100%|██████████| 14/14 [00:02<00:00,  5.03it/s]

Syn MAE=8.2031, MSE = 135.8121, RMSE=11.6538, R2=0.1840, CCp=0.4288, CCs=0.4322


### On syn_test

In [ ]:
# test_manual.py
import os
import math
import numpy as np
import pandas as pd
import argparse
import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm

from model_config import config
from dataset import SynergyDataset
from collator import SynergyDDI_collator
from model_ablation import SynergyDDI

# ---------------- Config ----------------
cfg = config['combine']

# ---------------- Manual Parameters ----------------
syn_csv="datasets/syn/syn_test.csv"
cell_expr_csv="datasets/ccle_expr_norm.csv"
emb_file1="datasets/ids/kpgt_base.npz"
emb_file2="datasets/ids/data_mol_repr_2000drug_conf1.npz"
ids_csv="datasets/ids_smiles/ids_smiles.csv"
selected_metrics=cfg.get("selected_metrics", None)
checkpoint = "/root/wd/Frame/draft_model/checkpoints/synddi/best_synMAE6.65_syn.pt"

# ---------------- Device ----------------
os.environ["CUDA_VISIBLE_DEVICES"] = '2,3'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



# ---------------- Dataset & Loader ----------------
test_dataset = SynergyDataset(
    syn_csv=syn_csv,
    cell_expr_csv=cell_expr_csv,
    emb_file1=emb_file1,
    emb_file2=emb_file2,
    ids_csv=ids_csv,
    selected_metrics=cfg.get("selected_metrics", None)
)

test_loader = DataLoader(
    test_dataset, batch_size=cfg['batch_size'],
    shuffle=False, collate_fn=SynergyDDI_collator,
    num_workers=cfg.get("num_workers", 4), pin_memory=True
)

# ---------------- Model ----------------
model = SynergyDDI(
    k_embed=cfg['k_embed'],
    u_embed=cfg['u_embed'],
    cell_expr_dim=cfg['cell_expr_dim'],
    dimreduct_dim=cfg['dimreduct_dim'],
    expr_hidden=cfg['expr_hidden'],
    regress_hidden=cfg['regress_hidden'],
    dropout_dimreduct=cfg['dropout_dimreduct'],
    dropout_cellfcn=cfg['dropout_cellfcn'],
    dropout_downstream=cfg['dropout_downstream'],
    mode=cfg['mode'],
    use_bn=cfg['use_bn']
)
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
model.to(device)

# Load checkpoint
state_dict = torch.load(checkpoint, map_location=device)
if hasattr(model, 'module'):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)
model.eval()

# ---------------- Metrics ----------------
all_syn_preds, all_syn_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", ncols=None):
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        syn_pred = model(
            batch_device["k_syn1"], batch_device["u_syn1"],
            batch_device["k_syn2"], batch_device["u_syn2"],
            batch_device["cell_expr"]
        )
        all_syn_preds.append(syn_pred.squeeze(-1).cpu().numpy())
        all_syn_labels.append(batch_device["syn_score"][:, 0].cpu().numpy())

# ---------------- Concatenate ----------------
syn_labels_all = np.concatenate(all_syn_labels)
syn_preds_all = np.concatenate(all_syn_preds)

# ---------------- Compute Metrics ----------------

def compute_mae_mse_rmse(target,prediction):
    error = []
    for i in range(len(target)):
        error.append(target[i] - prediction[i])
    squaredError = []
    absError = []
    for val in error:
        squaredError.append(val * val)  # target-prediction squared
        absError.append(abs(val))  # error absolute value
    mae=sum(absError)/len(absError)  # MAE
    mse=sum(squaredError)/len(squaredError)  # MSE
    RMSE=math.sqrt(mse)  # RMSE
    return mae,mse,RMSE

def compute_rsquared(X, Y):
    xBar = np.mean(X)
    yBar = np.mean(Y)
    SSR = 0
    varX = 0
    varY = 0
    for i in range(0, len(X)):
        diffXXBar = X[i] - xBar
        diffYYBar = Y[i] - yBar
        SSR += (diffXXBar * diffYYBar)
        varX += diffXXBar ** 2
        varY += diffYYBar ** 2

    SST = math.sqrt(varX * varY)
    r2=round((SSR / SST) ** 2,3)
    return r2


mae, mse, rmse = compute_mae_mse_rmse(syn_labels_all, syn_preds_all)
r2 = compute_rsquared(syn_labels_all, syn_preds_all)

#syn_mse = mean_squared_error(syn_labels_all, syn_preds_all)
#syn_mae = mean_absolute_error(syn_labels_all, syn_preds_all)
#syn_rmse = np.sqrt(syn_mse)
#syn_r2 = r2_score(syn_labels_all, syn_preds_all)
syn_ccp, _ = pearsonr(syn_labels_all, syn_preds_all)
syn_ccs, _ = spearmanr(syn_labels_all, syn_preds_all)

print(f"Syn MAE={mae:.4f}, MSE = {mse:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}, CCp={syn_ccp:.4f}, CCs={syn_ccs:.4f}")


Testing: 100%|██████████| 60/60 [00:04<00:00, 14.02it/s]


Syn MAE=5.2077, MSE = 59.1966, RMSE=7.6939, R2=0.7750, CCp=0.8803, CCs=0.7574


### KPGT Only

In [ ]:
import os
import numpy as np
import pandas as pd
import argparse
import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm
import math
from model_config import config
from dataset import Simple_CombDataset
from collator import simple_collator
from model_ablation import CombinedTwoModel_Simple

# ---------------- Config ----------------
cfg = config['kpgt']

# ---------------- Manual Parameters ----------------
moa_csv="datasets/moa/moa_test.csv"  # <- change to your test file
syn_csv="datasets/syn/my_oneil.csv"
cell_expr_csv="datasets/ccle_expr_norm.csv"
emb_file="datasets/ids/kpgt_base.npz"
ids_csv="datasets/ids_smiles/ids_smiles.csv"
selected_desc_labels=cfg.get("selected_desc_labels", None)
selected_metrics=cfg.get("selected_metrics", None)
checkpoint = "checkpoints/ablation/best_descAcc0.99_actionAcc0.99_synMAE6.84_kpgt_16_10.pt"

# ---------------- Device ----------------
os.environ["CUDA_VISIBLE_DEVICES"] = '0,1'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



# ---------------- Dataset & Loader ----------------
test_dataset = Simple_CombDataset(
    moa_csv=moa_csv,
    syn_csv=syn_csv,
    cell_expr_csv=cell_expr_csv,
    emb_file=emb_file,
    ids_csv=ids_csv,
    selected_desc_labels=cfg.get("selected_desc_labels", None),
    selected_metrics=cfg.get("selected_metrics", None)
)

test_loader = DataLoader(
    test_dataset, batch_size=cfg['batch_size'],
    shuffle=False, collate_fn=simple_collator,
    num_workers=cfg.get("num_workers", 4), pin_memory=True
)

# ---------------- Model ----------------
model = CombinedTwoModel_Simple(
    input_dim = cfg['input_dim'],
    num_desc_classes=cfg['num_desc_classes'],
    num_action_classes=cfg['num_action_classes'],
    cell_expr_dim=cfg['cell_expr_dim'],
    dimreduct_dim=cfg['dimreduct_dim'],
    expr_hidden=cfg['expr_hidden'],
    regress_hidden=cfg['regress_hidden'],
    lstm_hdims=cfg['lstm_hdims'],
    dropout_atten=cfg['dropout_atten'],
    dropout_reduct=cfg['dropout_dimreduct'],
    dropout_cellfcn=cfg['dropout_cellfcn'],
    dropout_downstream=cfg['dropout_downstream'],
    num_heads=cfg['num_heads'],
    use_bn=cfg['use_bn']
)
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
model.to(device)

# Load checkpoint
state_dict = torch.load(checkpoint, map_location=device)
if hasattr(model, 'module'):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)
model.eval()

# ---------------- Metrics ----------------
all_desc_preds, all_desc_labels = [], []
all_action_preds, all_action_labels = [], []
all_syn_preds, all_syn_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", ncols=None):
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        (desc_logits, action_logits), syn_pred = model(
            batch_device["moa1"], batch_device["moa2"],
            batch_device["syn1"], batch_device["syn2"],
            batch_device["cell_expr"]
        )

        all_desc_preds.append(torch.argmax(desc_logits, dim=-1).cpu().numpy())
        all_desc_labels.append(batch_device["desc_label"].cpu().numpy())
        all_action_preds.append(torch.argmax(action_logits, dim=-1).cpu().numpy())
        all_action_labels.append(batch_device["action_label"].cpu().numpy())
        all_syn_preds.append(syn_pred.squeeze(-1).cpu().numpy())
        all_syn_labels.append(batch_device["syn_score"][:, 0].cpu().numpy())

# ---------------- Concatenate ----------------
desc_labels_all = np.concatenate(all_desc_labels)
desc_preds_all = np.concatenate(all_desc_preds)
action_labels_all = np.concatenate(all_action_labels)
action_preds_all = np.concatenate(all_action_preds)
syn_labels_all = np.concatenate(all_syn_labels)
syn_preds_all = np.concatenate(all_syn_preds)

# ---------------- Compute Metrics ----------------
desc_acc = accuracy_score(desc_labels_all, desc_preds_all)
action_acc = accuracy_score(action_labels_all, action_preds_all)

per_class_acc = {}
for cls in np.unique(desc_labels_all):
    mask = desc_labels_all == cls
    per_class_acc[int(cls)] = accuracy_score(desc_labels_all[mask], desc_preds_all[mask])


def compute_mae_mse_rmse(target,prediction):
    error = []
    for i in range(len(target)):
        error.append(target[i] - prediction[i])
    squaredError = []
    absError = []
    for val in error:
        squaredError.append(val * val)  # target-prediction squared
        absError.append(abs(val))  # error absolute value
    mae=sum(absError)/len(absError)  # MAE
    mse=sum(squaredError)/len(squaredError)  # MSE
    RMSE=math.sqrt(mse)  # RMSE
    return mae,mse,RMSE

def compute_rsquared(X, Y):
    xBar = np.mean(X)
    yBar = np.mean(Y)
    SSR = 0
    varX = 0
    varY = 0
    for i in range(0, len(X)):
        diffXXBar = X[i] - xBar
        diffYYBar = Y[i] - yBar
        SSR += (diffXXBar * diffYYBar)
        varX += diffXXBar ** 2
        varY += diffYYBar ** 2

    SST = math.sqrt(varX * varY)
    r2=round((SSR / SST) ** 2,3)
    return r2


mae, mse, rmse = compute_mae_mse_rmse(syn_labels_all, syn_preds_all)
r2 = compute_rsquared(syn_labels_all, syn_preds_all)

#syn_mse = mean_squared_error(syn_labels_all, syn_preds_all)
#syn_mae = mean_absolute_error(syn_labels_all, syn_preds_all)
#syn_rmse = np.sqrt(syn_mse)
#syn_r2 = r2_score(syn_labels_all, syn_preds_all)
syn_ccp, _ = pearsonr(syn_labels_all, syn_preds_all)
syn_ccs, _ = spearmanr(syn_labels_all, syn_preds_all)

print(f"Desc Acc={desc_acc:.4f}, Action Acc={action_acc:.4f}")
print("Per-class desc accuracy:", per_class_acc)
print(f"Syn MAE={mae:.4f}, MSE = {mse:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}, CCp={syn_ccp:.4f}, CCs={syn_ccs:.4f}")


Testing:   0%|          | 0/14 [00:00<?, ?it/s]/root/miniconda3/envs/KPGT/lib/python3.7/site-packages/torch/nn/modules/rnn.py:692: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at  /opt/conda/conda-bld/pytorch_1634272168290/work/aten/src/ATen/native/cudnn/RNN.cpp:925.)
  self.dropout, self.training, self.bidirectional, self.batch_first)
Testing: 100%|██████████| 14/14 [00:03<00:00,  4.64it/s]


Desc Acc=0.9115, Action Acc=0.9107
Per-class desc accuracy: {0: 0.9392857142857143, 1: 0.9228159457167091, 2: 0.9069568928723858, 3: 0.7010213556174559}
Syn MAE=8.3339, MSE = 141.3807, RMSE=11.8904, R2=0.1810, CCp=0.4255, CCs=0.4374


In [ ]:
import os
import numpy as np
import pandas as pd
import argparse
import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score,precision_recall_fscore_support
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm
import math
from model_config import config
from dataset import Simple_CombDataset
from collator import simple_collator
from model_ablation import CombinedTwoModel_Simple

# ---------------- Config ----------------
cfg = config['kpgt']

# ---------------- Manual Parameters ----------------
moa_csv="datasets/moa/moa_test.csv"  # <- change to your test file
syn_csv="datasets/syn/my_oneil.csv"
cell_expr_csv="datasets/ccle_expr_norm.csv"
emb_file="datasets/ids/kpgt_base.npz"
ids_csv="datasets/ids_smiles/ids_smiles.csv"
selected_desc_labels=cfg.get("selected_desc_labels", None)
selected_metrics=cfg.get("selected_metrics", None)
checkpoint = "checkpoints/ablation/best_descAcc0.99_actionAcc0.99_synMAE6.84_kpgt_16_10.pt"

# ---------------- Device ----------------
os.environ["CUDA_VISIBLE_DEVICES"] = '0,1'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



# ---------------- Dataset & Loader ----------------
test_dataset = Simple_CombDataset(
    moa_csv=moa_csv,
    syn_csv=syn_csv,
    cell_expr_csv=cell_expr_csv,
    emb_file=emb_file,
    ids_csv=ids_csv,
    selected_desc_labels=cfg.get("selected_desc_labels", None),
    selected_metrics=cfg.get("selected_metrics", None)
)

test_loader = DataLoader(
    test_dataset, batch_size=cfg['batch_size'],
    shuffle=False, collate_fn=simple_collator,
    num_workers=cfg.get("num_workers", 4), pin_memory=True
)

# ---------------- Model ----------------
model = CombinedTwoModel_Simple(
    input_dim = cfg['input_dim'],
    num_desc_classes=cfg['num_desc_classes'],
    num_action_classes=cfg['num_action_classes'],
    cell_expr_dim=cfg['cell_expr_dim'],
    dimreduct_dim=cfg['dimreduct_dim'],
    expr_hidden=cfg['expr_hidden'],
    regress_hidden=cfg['regress_hidden'],
    lstm_hdims=cfg['lstm_hdims'],
    dropout_atten=cfg['dropout_atten'],
    dropout_reduct=cfg['dropout_dimreduct'],
    dropout_cellfcn=cfg['dropout_cellfcn'],
    dropout_downstream=cfg['dropout_downstream'],
    num_heads=cfg['num_heads'],
    use_bn=cfg['use_bn']
)
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
model.to(device)

# Load checkpoint
state_dict = torch.load(checkpoint, map_location=device)
if hasattr(model, 'module'):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)
model.eval()

# ---------------- Metrics ----------------
all_desc_preds, all_desc_labels = [], []
all_action_preds, all_action_labels = [], []
all_syn_preds, all_syn_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", ncols=None):
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        (desc_logits, action_logits), syn_pred = model(
            batch_device["moa1"], batch_device["moa2"],
            batch_device["syn1"], batch_device["syn2"],
            batch_device["cell_expr"]
        )

        all_desc_preds.append(torch.argmax(desc_logits, dim=-1).cpu().numpy())
        all_desc_labels.append(batch_device["desc_label"].cpu().numpy())
        all_action_preds.append(torch.argmax(action_logits, dim=-1).cpu().numpy())
        all_action_labels.append(batch_device["action_label"].cpu().numpy())
        all_syn_preds.append(syn_pred.squeeze(-1).cpu().numpy())
        all_syn_labels.append(batch_device["syn_score"][:, 0].cpu().numpy())

# ---------------- Concatenate ----------------
desc_labels_all = np.concatenate(all_desc_labels)
desc_preds_all = np.concatenate(all_desc_preds)
action_labels_all = np.concatenate(all_action_labels)
action_preds_all = np.concatenate(all_action_preds)
syn_labels_all = np.concatenate(all_syn_labels)
syn_preds_all = np.concatenate(all_syn_preds)

# ---------------- Compute Accuracy ----------------
desc_acc = accuracy_score(desc_labels_all, desc_preds_all)
action_acc = accuracy_score(action_labels_all, action_preds_all)

# ---------------- Compute Precision/Recall/F1 ----------------
# Desc
p_desc, r_desc, f_desc, support_desc = precision_recall_fscore_support(
    desc_labels_all, desc_preds_all, labels=np.unique(desc_labels_all), zero_division=0
)
per_class_desc = {}
for i, cls in enumerate(np.unique(desc_labels_all)):
    per_class_desc[int(cls)] = {
        "precision": p_desc[i],
        "recall": r_desc[i],
        "f1": f_desc[i],
        "support": support_desc[i],
        "accuracy": (desc_preds_all[desc_labels_all==cls]==cls).mean()
    }

# Action
p_action, r_action, f_action, support_action = precision_recall_fscore_support(
    action_labels_all, action_preds_all, labels=np.unique(action_labels_all), zero_division=0
)
per_class_action = {}
for i, cls in enumerate(np.unique(action_labels_all)):
    per_class_action[int(cls)] = {
        "precision": p_action[i],
        "recall": r_action[i],
        "f1": f_action[i],
        "support": support_action[i],
        "accuracy": (action_preds_all[action_labels_all==cls]==cls).mean()
    }

# Overall macro metrics
desc_precision, desc_recall, desc_f1, _ = precision_recall_fscore_support(
    desc_labels_all, desc_preds_all, average='macro', zero_division=0
)
action_precision, action_recall, action_f1, _ = precision_recall_fscore_support(
    action_labels_all, action_preds_all, average='macro', zero_division=0
)

# ---------------- Regression Metrics ----------------
def compute_mae_mse_rmse(target, prediction):
    error = target - prediction
    mae = np.mean(np.abs(error))
    mse = np.mean(error**2)
    rmse = np.sqrt(mse)
    return mae, mse, rmse

def compute_rsquared(X, Y):
    xBar = np.mean(X)
    yBar = np.mean(Y)
    SSR = np.sum((X - xBar) * (Y - yBar))
    varX = np.sum((X - xBar)**2)
    varY = np.sum((Y - yBar)**2)
    SST = np.sqrt(varX * varY)
    if SST == 0:
        return 0.0
    return round((SSR / SST)**2, 3)

mae, mse, rmse = compute_mae_mse_rmse(syn_labels_all, syn_preds_all)
r2 = compute_rsquared(syn_labels_all, syn_preds_all)
syn_ccp, _ = pearsonr(syn_labels_all, syn_preds_all)
syn_ccs, _ = spearmanr(syn_labels_all, syn_preds_all)

# ---------------- Print Results ----------------
print("="*60)
print(f"Desc Acc={desc_acc:.4f}, Precision={desc_precision:.4f}, Recall={desc_recall:.4f}, F1={desc_f1:.4f}")
print(f"Action Acc={action_acc:.4f}, Precision={action_precision:.4f}, Recall={action_recall:.4f}, F1={action_f1:.4f}")

print("\nPer-class Desc Metrics:")
for cls, metrics in per_class_desc.items():
    print(f"  Class {cls}: Acc={metrics['accuracy']:.4f}, "
          f"Prec={metrics['precision']:.4f}, Rec={metrics['recall']:.4f}, F1={metrics['f1']:.4f}, "
          f"Support={metrics['support']}")

print("\nPer-class Action Metrics:")
for cls, metrics in per_class_action.items():
    print(f"  Class {cls}: Acc={metrics['accuracy']:.4f}, "
          f"Prec={metrics['precision']:.4f}, Rec={metrics['recall']:.4f}, F1={metrics['f1']:.4f}, "
          f"Support={metrics['support']}")

print("\nRegression Metrics (Synergy Prediction):")
print(f"  MAE={mae:.4f}, MSE={mse:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}, CCp={syn_ccp:.4f}, CCs={syn_ccs:.4f}")
print("="*60)

Testing:   0%|          | 0/27 [00:00<?, ?it/s]/root/miniconda3/envs/KPGT/lib/python3.7/site-packages/torch/nn/modules/rnn.py:692: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at  /opt/conda/conda-bld/pytorch_1634272168290/work/aten/src/ATen/native/cudnn/RNN.cpp:925.)
  self.dropout, self.training, self.bidirectional, self.batch_first)
Testing: 100%|██████████| 27/27 [00:02<00:00, 10.88it/s]

Desc Acc=0.9109, Precision=0.8721, Recall=0.8670, F1=0.8695
Action Acc=0.9112, Precision=0.9111, Recall=0.8979, F1=0.9037

Per-class Desc Metrics:
  Class 0: Acc=0.9389, Prec=0.9288, Rec=0.9389, F1=0.9338, Support=7560
  Class 1: Acc=0.9228, Prec=0.9339, Rec=0.9228, F1=0.9283, Support=2358
  Class 2: Acc=0.9052, Prec=0.9299, Rec=0.9052, F1=0.9174, Support=2343
  Class 3: Acc=0.7010, Prec=0.6959, Rec=0.7010, F1=0.6984, Support=1077

Per-class Action Metrics:
  Class 0: Acc=0.8453, Prec=0.9107, Rec=0.8453, F1=0.8768, Support=4983
  Class 1: Acc=0.9506, Prec=0.9115, Rec=0.9506, F1=0.9306, Support=8355

Regression Metrics (Synergy Prediction):
  MAE=8.3339, MSE=141.3807, RMSE=11.8904, R2=0.1810, CCp=0.4255, CCs=0.4374


### Unimol Only

In [12]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import math
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from scipy.stats import pearsonr, spearmanr

from model_config import config
from dataset import Simple_CombDataset
from collator import simple_collator
from model_ablation import CombinedTwoModel_Simple

# ---------------- Config ----------------
cfg = config['unimol']

# ---------------- Manual Parameters ----------------
moa_csv = "datasets/moa/moa_test.csv"
syn_csv = "datasets/syn/my_oneil.csv"
cell_expr_csv = "datasets/ccle_expr_norm.csv"
emb_file = "datasets/ids/data_mol_repr_2000drug_conf1.npz"
ids_csv="datasets/ids_smiles/ids_smiles.csv"
selected_desc_labels = cfg.get("selected_desc_labels", None)
selected_metrics = cfg.get("selected_metrics", None)
checkpoint = "checkpoints/ablation/best_descAcc0.94_actionAcc0.96_synMAE6.91_unimol_16_10.pt"

# ---------------- Device ----------------
os.environ["CUDA_VISIBLE_DEVICES"] = '2,3'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------- Dataset & Loader ----------------
test_dataset = Simple_CombDataset(
    moa_csv=moa_csv,
    syn_csv=syn_csv,
    cell_expr_csv=cell_expr_csv,
    emb_file=emb_file,
    ids_csv=ids_csv,
    selected_desc_labels=selected_desc_labels,
    selected_metrics=selected_metrics
)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg['batch_size'],
    shuffle=False,
    collate_fn=simple_collator,
    num_workers=cfg.get("num_workers", 4),
    pin_memory=True
)

# ---------------- Model ----------------
model = CombinedTwoModel_Simple(
    input_dim=cfg['input_dim'],
    num_desc_classes=cfg['num_desc_classes'],
    num_action_classes=cfg['num_action_classes'],
    cell_expr_dim=cfg['cell_expr_dim'],
    dimreduct_dim=cfg['dimreduct_dim'],
    expr_hidden=cfg['expr_hidden'],
    regress_hidden=cfg['regress_hidden'],
    lstm_hdims=cfg['lstm_hdims'],
    dropout_atten=cfg['dropout_atten'],
    dropout_reduct=cfg['dropout_dimreduct'],
    dropout_cellfcn=cfg['dropout_cellfcn'],
    dropout_downstream=cfg['dropout_downstream'],
    num_heads=cfg['num_heads'],
    use_bn=cfg['use_bn']
)
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
model.to(device)

# ---------------- Load Checkpoint ----------------
state_dict = torch.load(checkpoint, map_location=device)
if hasattr(model, 'module'):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)
model.eval()

# ---------------- Metrics Storage ----------------
all_desc_preds, all_desc_labels = [], []
all_action_preds, all_action_labels = [], []
all_syn_preds, all_syn_labels = [], []

# ---------------- Test Loop ----------------
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", ncols=None):
        batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        (desc_logits, action_logits), syn_pred = model(
            batch_device["moa1"], batch_device["moa2"],
            batch_device["syn1"], batch_device["syn2"],
            batch_device["cell_expr"]
        )

        all_desc_preds.append(torch.argmax(desc_logits, dim=-1).cpu().numpy())
        all_desc_labels.append(batch_device["desc_label"].cpu().numpy())
        all_action_preds.append(torch.argmax(action_logits, dim=-1).cpu().numpy())
        all_action_labels.append(batch_device["action_label"].cpu().numpy())
        all_syn_preds.append(syn_pred.squeeze(-1).cpu().numpy())
        all_syn_labels.append(batch_device["syn_score"][:, 0].cpu().numpy())

# ---------------- Concatenate ----------------
desc_labels_all = np.concatenate(all_desc_labels)
desc_preds_all = np.concatenate(all_desc_preds)
action_labels_all = np.concatenate(all_action_labels)
action_preds_all = np.concatenate(all_action_preds)
syn_labels_all = np.concatenate(all_syn_labels)
syn_preds_all = np.concatenate(all_syn_preds)

# ---------------- Compute Accuracy ----------------
desc_acc = accuracy_score(desc_labels_all, desc_preds_all)
action_acc = accuracy_score(action_labels_all, action_preds_all)

# ---------------- Compute Precision/Recall/F1 ----------------
# Desc
p_desc, r_desc, f_desc, support_desc = precision_recall_fscore_support(
    desc_labels_all, desc_preds_all, labels=np.unique(desc_labels_all), zero_division=0
)
per_class_desc = {}
for i, cls in enumerate(np.unique(desc_labels_all)):
    per_class_desc[int(cls)] = {
        "precision": p_desc[i],
        "recall": r_desc[i],
        "f1": f_desc[i],
        "support": support_desc[i],
        "accuracy": (desc_preds_all[desc_labels_all==cls]==cls).mean()
    }

# Action
p_action, r_action, f_action, support_action = precision_recall_fscore_support(
    action_labels_all, action_preds_all, labels=np.unique(action_labels_all), zero_division=0
)
per_class_action = {}
for i, cls in enumerate(np.unique(action_labels_all)):
    per_class_action[int(cls)] = {
        "precision": p_action[i],
        "recall": r_action[i],
        "f1": f_action[i],
        "support": support_action[i],
        "accuracy": (action_preds_all[action_labels_all==cls]==cls).mean()
    }

# Overall macro metrics
desc_precision, desc_recall, desc_f1, _ = precision_recall_fscore_support(
    desc_labels_all, desc_preds_all, average='macro', zero_division=0
)
action_precision, action_recall, action_f1, _ = precision_recall_fscore_support(
    action_labels_all, action_preds_all, average='macro', zero_division=0
)

# ---------------- Regression Metrics ----------------
def compute_mae_mse_rmse(target, prediction):
    error = target - prediction
    mae = np.mean(np.abs(error))
    mse = np.mean(error**2)
    rmse = np.sqrt(mse)
    return mae, mse, rmse

def compute_rsquared(X, Y):
    xBar = np.mean(X)
    yBar = np.mean(Y)
    SSR = np.sum((X - xBar) * (Y - yBar))
    varX = np.sum((X - xBar)**2)
    varY = np.sum((Y - yBar)**2)
    SST = np.sqrt(varX * varY)
    if SST == 0:
        return 0.0
    return round((SSR / SST)**2, 3)

mae, mse, rmse = compute_mae_mse_rmse(syn_labels_all, syn_preds_all)
r2 = compute_rsquared(syn_labels_all, syn_preds_all)
syn_ccp, _ = pearsonr(syn_labels_all, syn_preds_all)
syn_ccs, _ = spearmanr(syn_labels_all, syn_preds_all)

# ---------------- Print Results ----------------
print("="*60)
print(f"Desc Acc={desc_acc:.4f}, Precision={desc_precision:.4f}, Recall={desc_recall:.4f}, F1={desc_f1:.4f}")
print(f"Action Acc={action_acc:.4f}, Precision={action_precision:.4f}, Recall={action_recall:.4f}, F1={action_f1:.4f}")

print("\nPer-class Desc Metrics:")
for cls, metrics in per_class_desc.items():
    print(f"  Class {cls}: Acc={metrics['accuracy']:.4f}, "
          f"Prec={metrics['precision']:.4f}, Rec={metrics['recall']:.4f}, F1={metrics['f1']:.4f}, "
          f"Support={metrics['support']}")

print("\nPer-class Action Metrics:")
for cls, metrics in per_class_action.items():
    print(f"  Class {cls}: Acc={metrics['accuracy']:.4f}, "
          f"Prec={metrics['precision']:.4f}, Rec={metrics['recall']:.4f}, F1={metrics['f1']:.4f}, "
          f"Support={metrics['support']}")

print("\nRegression Metrics (Synergy Prediction):")
print(f"  MAE={mae:.4f}, MSE={mse:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}, CCp={syn_ccp:.4f}, CCs={syn_ccs:.4f}")
print("="*60)


Testing:   0%|          | 0/14 [00:00<?, ?it/s]/root/miniconda3/envs/KPGT/lib/python3.7/site-packages/torch/nn/modules/rnn.py:692: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at  /opt/conda/conda-bld/pytorch_1634272168290/work/aten/src/ATen/native/cudnn/RNN.cpp:925.)
  self.dropout, self.training, self.bidirectional, self.batch_first)
Testing: 100%|██████████| 14/14 [00:02<00:00,  5.63it/s]

Desc Acc=0.8766, Precision=0.8164, Recall=0.8603, F1=0.8350
Action Acc=0.8915, Precision=0.8877, Recall=0.8787, F1=0.8828

Per-class Desc Metrics:
  Class 0: Acc=0.8730, Prec=0.9346, Rec=0.8730, F1=0.9027, Support=7560
  Class 1: Acc=0.9177, Prec=0.8779, Rec=0.9177, F1=0.8974, Support=2358
  Class 2: Acc=0.9087, Prec=0.8878, Rec=0.9087, F1=0.8981, Support=2343
  Class 3: Acc=0.7419, Prec=0.5655, Rec=0.7419, F1=0.6418, Support=1077

Per-class Action Metrics:
  Class 0: Acc=0.8282, Prec=0.8747, Rec=0.8282, F1=0.8508, Support=4983
  Class 1: Acc=0.9293, Prec=0.9007, Rec=0.9293, F1=0.9148, Support=8355

Regression Metrics (Synergy Prediction):
  MAE=8.6664, MSE=141.7727, RMSE=11.9068, R2=0.1730, CCp=0.4156, CCs=0.4107
